# 02 — Exploration des données formattées (Yahoo Finance)

Lecture du fichier `formatted/yahoofinance/wti.parquet` depuis S3 (LocalStack) pour vérifier le résultat du script `clean_yfinance.py`.

In [2]:
import io
import os
import boto3
import pandas as pd

# ── Connexion S3 LocalStack ──
os.environ["AWS_ACCESS_KEY_ID"] = "test"
os.environ["AWS_SECRET_ACCESS_KEY"] = "test"
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

s3 = boto3.client("s3", endpoint_url="http://localhost:4566")

# ── Lecture du parquet formatté (dossier Spark) ──
# Spark écrit un dossier wti.parquet/ contenant _SUCCESS + part-*.parquet
prefix = "formatted/yahoofinance/wti.parquet/"
objs = s3.list_objects_v2(Bucket="datalake", Prefix=prefix)
parquet_key = [o["Key"] for o in objs.get("Contents", []) if o["Key"].endswith(".parquet")][0]

response = s3.get_object(Bucket="datalake", Key=parquet_key)
df = pd.read_parquet(io.BytesIO(response["Body"].read()), engine="pyarrow")

print(f"Shape : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nTypes :\n{df.dtypes}")
print(f"\nPlage : {df['Datetime'].min()} → {df['Datetime'].max()}")
df.head()

Shape : 155 lignes × 6 colonnes

Types :
Datetime    datetime64[ns]
Close              float64
High               float64
Low                float64
Open               float64
Volume               int64
dtype: object

Plage : 2026-02-24 05:00:00 → 2026-02-25 20:30:00


,Datetime,Close,High,Low,Open,Volume
0,2026-02-24 05:00:00,66.750000,66.800003,66.610001,66.790001,364
1,2026-02-24 05:15:00,66.709999,66.739998,66.690002,66.739998,318
2,2026-02-24 05:30:00,66.919998,66.959999,66.690002,66.699997,915
3,2026-02-24 05:45:00,66.849998,66.940002,66.820000,66.919998,724
4,2026-02-24 06:00:00,66.769997,66.860001,66.750000,66.849998,860


In [3]:
# ── Vérification qualité ──
print("Valeurs manquantes :")
print(df.isnull().sum())
print(f"\nDoublons sur Datetime : {df.duplicated(subset=['Datetime']).sum()}")
print(f"\nStatistiques :")
df.describe().round(2)

Valeurs manquantes :
Datetime    0
Close       0
High        0
Low         0
Open        0
Volume      0
dtype: int64

Doublons sur Datetime : 0

Statistiques :


,Datetime,Close,High,Low,Open,Volume
count,155,155.00,155.00,155.00,155.00,155.00
mean,2026-02-25 00:48:40.645161216,66.08,66.18,65.98,66.09,3551.41
min,2026-02-24 05:00:00,65.32,65.46,65.12,65.31,233.00
25%,2026-02-24 14:37:30,65.87,65.95,65.76,65.88,940.50
50%,2026-02-25 01:15:00,66.04,66.13,65.97,66.05,1876.00
75%,2026-02-25 10:52:30,66.30,66.42,66.22,66.31,5044.00
max,2026-02-25 20:30:00,67.02,67.15,66.86,67.02,21717.00
std,NaN,0.38,0.38,0.39,0.38,3871.79


In [4]:
# ── Aperçu dernières lignes ──
print("Dernières lignes :")
df.tail()

Dernières lignes :


,Datetime,Close,High,Low,Open,Volume
150,2026-02-25 19:30:00,65.650002,65.720001,65.449997,65.470001,7229
151,2026-02-25 19:45:00,65.720001,65.750000,65.620003,65.639999,1486
152,2026-02-25 20:00:00,65.680000,65.730003,65.660004,65.720001,1476
153,2026-02-25 20:15:00,65.599998,65.690002,65.589996,65.680000,1178
154,2026-02-25 20:30:00,65.529999,65.599998,65.449997,65.589996,1243
